In [ ]:
#1)Import required libraries
import pandas as pd
import requests
import json
import csv
import sys
import contextlib
import re
import os
recon3D_dir = '../datasets/network_models/'
synonym_mapping_dir = '../datasets/naming/'

In [ ]:
#2) Read Recon3d.json
recon3d_file = open(os.path.join(recon3D_dir,"Recon3D.json"))
recon3d_json = json.load(recon3d_file)

In [ ]:
#3) Read synonym-mapping.json
synonym_mapping_file = open(os.path.join(synonym_mapping_dir,"synonym-mapping.json"))
synonym_mapping_file_json = json.load(synonym_mapping_file)

In [ ]:
#Function to search the given metabolite name in recond3d_json to check whether recon_id info exists or not
def find_reconids(name):
    all_metabolites = recon3d_json["metabolites"]
    all_results = []
    for metabolite in all_metabolites:
        if(metabolite["name"] == name):
            all_results.append(metabolite["id"])
        #print(metabolite["name"])
    return []


In [ ]:
#Function to search the given metabolite name in synonym_mapping.json to check whether synonym info exists or not
def find_in_synonym_mapping(name):
    if name in synonym_mapping_file_json.keys():
        return synonym_mapping_file_json.get(name)
    else:
        return ""


In [ ]:
#The most efficent version of format name functions
#Function to clean the dirty name data for proper search at RefMet
def format_name_newest3(name):

    #if there is [ at the begining remove it
    name = name.lstrip(" [")
    #we are sure that data contains , and ' at both the begining and end, so remove them
    name = name.lstrip(" ',").rstrip(" ',")
    #remove the formulas which is in between []
    name = re.sub(r"[\[].*?[\]]", '', name)

    #check the part after + consist only letters
    before, sep, after = name.partition("+")
    after = re.sub(r"\s+", '', after) #remove whitespaces for proper check
    if(not after.isalpha()):
        #if no, only take the part before +
        name = before

    #remove all -'s from begining and end
    name = re.sub(r"^[-]+|[-]+$", "", name)
    
    #divide name wrt first _
    before, sep, after = name.partition("_")
    #if the length of the part before is longer than the part after, take only the part before
    if(len(before) > len(after)):
        name = before

    #divide name wrt first (
    before, sep, after = name.partition("(")
    #if the length of the part before is longer than the part after, take only the part before
    if(len(before) > len(after)):
        name = before
        #remove all non alphanumeric characters from both the begining and end
        name = re.sub(r"^\W+|\W+$", "", name)

    return name

In [ ]:
#NO NEED TO COMPILE, JUST FOR EXPERIMENT
#Another version of format name function
#Function to clean the dirty name data for proper search at RefMet
def format_name_newest2(name):

    #we are sure that data contains , and ' at both the begining and end
    name = name.lstrip(" ',").rstrip(" ',")
    
    #remove the formulas which is in between []
    name = re.sub(r"[\[].*?[\]]", '', name)

    #remove the part after +
    name = re.sub(r"\+.*", '', name)

    name = re.sub(r"^[-]+|[-]+$", "", name)
    
    #remove all non alphanumeric characters from both the begining and end
    #name = re.sub(r"^\W+|\W+$", "", name)
    #print(name)
    before, sep, after = name.partition("_")
    if(len(before) > len(after)):
        name = before

    before, sep, after = name.partition("(")
    if(len(before) > len(after)):
        name = before
        name = re.sub(r"^\W+|\W+$", "", name)

    return name

In [ ]:
#NO NEED TO COMPILE, JUST FOR EXPERIMENT
#Another version of format name function
#Function to clean the dirty name data for proper search at RefMet
def format_name_newest(name):

    name = re.sub(r"^\W+|\W+$", "", name)
    
    before, sep, after = name.partition("_")
        
    if len(after) > 0:
        if after.isnumeric() or '_' in after:
            ##add same for before
            a,b,c = before.partition("(")
            name = re.sub(r"^\W+|\W+$", "", a)
        else:
            #print(after)
            a,b,c = after.partition("(")
            name = re.sub(r"^\W+|\W+$", "", a)
        return name
    else:
        a,b,c = before.partition("(")
        name = re.sub(r"^\W+|\W+$", "", a)
        #print(a)
        return name

In [ ]:
#NO NEED TO COMPILE, JUST FOR EXPERIMENT
#Another version of format name function
#Function to clean the dirty name data for proper search at RefMet
def format_name_new(name):
        name = name.lstrip(" ',_+-/(*)}{[]\"\;@.?!").rstrip(" ',_+-/(*)}{[]\"\;@.?!")
        string ="_"
        before, sep, after = name.partition("_")
        if len(after) > 0:
                if after.isnumeric():
                        ##add same for before
                        name = before.lstrip(" ',_+-/(*)}{[]\"\;@.?!").rstrip(" ',_+-/(*)}{[]\"\;@.?!")
                        #regular expression
                else:
                        name = after.lstrip(" ',_+-/(*)}{[]\"\;@.?!").rstrip(" ',_+-/(*)}{[]\"\;@.?!")
        return name


In [ ]:
#NO NEED TO COMPILE, JUST FOR EXPERIMENT
#The first version of format name function
#Function to clean the dirty name data for proper search at RefMet
def format_name_old(name):
        string ="_"
        before, sep, after = name.partition("_")
        
        if len(after) > 0:
                name = after
        return name

In [ ]:
# 4) To work on all datasets

#For debugging, uncomment all print lines

#I put all data files into a folder named "datasets"
for file in os.listdir("datasets"):
    #for every file in the folder
    #print(file)

    #read the file
    unmapped_metabolites_df = pd.read_excel(file)
    row = 0
    #read the data row by row to a list from file
    unmapped_metabolites = []
    for row in range(unmapped_metabolites_df.size):
        name = unmapped_metabolites_df._get_value(row,"name")
        unmapped_metabolites.append(name) 
        row += 1
       
    founded_count = 0 #contains the info of how many enchancements will be done
    #for all unmapped metabolites
    for x in unmapped_metabolites:
        #clean the name
        formatted_x = format_name_newest3(x)
        #check for RefMet match
        url_for_name_match = "https://www.metabolomicsworkbench.org/rest/refmet/match/" + formatted_x #to get rid of useless characters
        name_match_response = requests.get(url_for_name_match)
        name_match_data = name_match_response.content
        name_match_data_json = json.loads(name_match_data)
        #if there is at least one RefMet match
        if(name_match_data_json != []):
            refmet_name = name_match_data_json.get("refmet_name")
            
            #FIRST CHECK RECON3D
            #Recon3D may return more than one reconids
            resulted_synonyms_from_recon3d = find_reconids(refmet_name)
            resulted_synonym = ""
            if len(resulted_synonyms_from_recon3d) == 1:
                resulted_synonym = resulted_synonyms_from_recon3d[0]
            elif len(resulted_synonyms_from_recon3d) > 1:
                for s in resulted_synonyms_from_recon3d:
                    if(re.search("_c$",s)):
                        resulted_synonym = s
                        break
                if resulted_synonym == "":
                    resulted_synonym = resulted_synonyms_from_recon3d[0]
                    

            #If nothing obtained from RECON3D check
            #THEN CHECK SYNOYNM_MAPPING
            if resulted_synonym == "":
                resulted_synonym = find_in_synonym_mapping(refmet_name.lower())
        
            #If a match obtained until this point
            if resulted_synonym != "":
                #print(x ," ==> ", formatted_x ," ==> " , refmet_name," ==> ",resulted_synonym)
                #UPDATE EXISTING SYNOYM MAPPING
                synonym_mapping_file_json.update({x:resulted_synonym})
                if not(formatted_x in synonym_mapping_file_json.keys()):
                    synonym_mapping_file_json.update({formatted_x:resulted_synonym})
                if not(refmet_name in synonym_mapping_file_json.keys()):
                    synonym_mapping_file_json.update({refmet_name:resulted_synonym})
                founded_count += 1
   
    #print(founded_count)

#WRITE THE UPDATED VERSIION TO A NEW FILE
with open("synonym-mapping-new.json", "w") as outfile:
    json.dump(synonym_mapping_file_json, outfile)


In [ ]:
#IDENTICAL TO PREVIOUS ONE ONLY FOR LOOP DOESNT EXIST
#To debug single dataset

unmapped_metabolites_df = pd.read_excel("ST000641_name.xlsx")#CHANGE THE FILE NAME BY HAND
row = 0
unmapped_metabolites = []
for row in range(unmapped_metabolites_df.size):
    name = unmapped_metabolites_df._get_value(row,"name")
    unmapped_metabolites.append(name) 
    row += 1
       
founded_count = 0
for x in unmapped_metabolites:
    formatted_x = format_name_newest3(x)
    url_for_name_match = "https://www.metabolomicsworkbench.org/rest/refmet/match/" + formatted_x #to get rid of useless characters
    name_match_response = requests.get(url_for_name_match)
    name_match_data = name_match_response.content
    name_match_data_json = json.loads(name_match_data)
 
    if(name_match_data_json != []):
        
        refmet_name = name_match_data_json.get("refmet_name")

        #FIRST CHECK RECON3D
        resulted_synonyms_from_recon3d = find_reconids(refmet_name)
        resulted_synonym = ""
        if len(resulted_synonyms_from_recon3d) == 1:
            resulted_synonym = resulted_synonyms_from_recon3d[0]
        elif len(resulted_synonyms_from_recon3d) > 1:
            for s in resulted_synonyms_from_recon3d:
                if(re.search("_c$",s)):
                    resulted_synonym = s
                    break
            if resulted_synonym == "":
                resulted_synonym = resulted_synonyms_from_recon3d[0]
                    


        if resulted_synonym == "":
            #CHECK SYNOYNM_MAPPING
            resulted_synonym = find_in_synonym_mapping(refmet_name.lower())
        
        
        #print(x ," ==> ", formatted_x ," ==> " , refmet_name," ==> ",resulted_synonym)
            
        if resulted_synonym != "":
            #print("-----------FOUNDED----------")
            founded_count += 1
        
print(founded_count)
